# Fly-robot: a robot arm run by a piece of the real fly brain

The robot's camera is wired into the fly's **eye** (photoreceptors), its joints into the fly's
**body-sense** neurons (ascending neurons), and its motors are driven by the fly's **command**
neurons (descending neurons). In between: ~2,500 neurons on the real pathways through the
optic lobe and central brain (FlyWire connectome). Only the fly brain is trained (PPO).

The brain sees the tray through a 16x16 wrist camera (this frame + the previous one). It is
**not** told where the ball is.

**Kaggle settings:** Internet ON. Accelerator: **None (CPU)** is fine: the brain is stored as a
sparse matrix, which is fast on CPU (~3-4 h per 1M steps expected). Use **Save & Run All (Commit)**
for long runs.

In [ ]:
# ===== Settings =====
REPO_URL = "https://github.com/nihal4/connectome-tray-balance.git"
BRANCH = "main"

ENV = "tray_vision"        # tray_vision -> maze_fixed3_vision -> maze3_vision -> maze3h_vision
TOTAL_STEPS = 1_000_000
N_NEURONS = 2500           # brain size (eye 512 + body 64 + commands 64 + pathway neurons)
SEED = 0
DEVICE = "cpu"             # "cuda" should also work but is untested for the sparse brain

In [ ]:
import os, subprocess, sys, glob, shutil
WORK = "/kaggle/working"
REPO = f"{WORK}/connectome-tray-balance"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3 psutil imageio imageio-ffmpeg

## 1. Download the connectome + cell types, build the fly-robot brain

In [ ]:
!python scripts/download_connectome.py --out data/raw
!python scripts/build_flybrain.py --n {N_NEURONS}
BRAIN = sorted(glob.glob(f"data/flybrain_cam16_N*.npz"))[-1]
print("brain file:", BRAIN)

`eye_to_command_hops` shows how many synapses a signal crosses from the eye to the command
neurons inside this brain (it should be a small number like 3-5 for all of them).

## 2. Check the task (the camera alone is enough to solve it)

In [ ]:
!python scripts/check_mac.py --env {ENV} --subset {BRAIN}

## 3. Train the fly brain

In [ ]:
import re
KEEP = re.compile(r"^\[|Eval num|Episode length|Success rate|New best")
def train(env=ENV, extra=""):
    cmd = (f"{sys.executable} -u scripts/train.py --env {env} --policy connectome --subset {BRAIN} "
           f"--profile kaggle --total_steps {TOTAL_STEPS} --seed {SEED} --device {DEVICE} "
           f"--runs_dir {WORK}/runs {extra}")
    p = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if KEEP.search(line):
            print(line, end="", flush=True)
    if p.wait() != 0:
        print(f"!! exited with code {p.returncode}")

train()

To continue a run later (e.g. another 1M steps): set `TOTAL_STEPS = 2_000_000` and run
`train(extra="--resume")`.

## 4. Results

In [ ]:
run = f"{WORK}/runs/{ENV}-connectome-real-s{SEED}"
!python scripts/plot_runs.py {run} --out {WORK}/curves_{ENV}.png
from IPython.display import Image, Video, display
display(Image(f"{WORK}/curves_{ENV}.png"))

In [ ]:
# Tests it never saw in training: camera noise, blind (ball invisible), heavy ball, pushes
!python scripts/evaluate.py --run {run} --robustness --episodes 10

In [ ]:
# Video, with what the fly brain sees (the camera image) in the top-left corner
os.environ["MUJOCO_GL"] = "egl"
vid = f"{WORK}/{ENV}_flyrobot.mp4"
!python scripts/evaluate.py --run {run} --episodes 1 --video {vid}
if os.path.exists(vid):
    display(Video(vid, embed=True, width=480))

In [ ]:
shutil.make_archive(f"{WORK}/runs_export", "zip", f"{WORK}/runs")
print("download runs_export.zip and the .mp4 from the Output panel")